# Notebook 05: Building a Decentralized Voting Application

In this final notebook of the core series, we will build a more complex Decentralized Application (DApp) - a Voting System.

This smart contract will introduce us to:
1. **Structs:** Grouping multiple variables together.
2. **Access Control:** Ensuring only certain users (like a chairperson) can perform specific actions.
3. **Complex State Logic:** Handling delegations (allowing someone else to vote for you).

We will again use the Ape Framework structure to compile and test our application.

In [ ]:
# Ensure Ape is installed
!pip install eth-ape ape-vyper
!ape plugins install vyper

!mkdir -p voting_app/contracts
!mkdir -p voting_app/tests
!touch voting_app/ape-config.yaml

### 1. The Voting Contract

Let's write `DelegateVotingApp.vy`. This contract allows a `chairperson` to give the right to vote to users. Users can either vote directly for a proposal or `delegate` their vote to someone they trust.

In [ ]:
%%writefile voting_app/contracts/DelegateVotingApp.vy
# @version ^0.3.0

# 1. Structs
struct Voter:
    weight: uint256
    voted: bool
    delegate: address
    vote: uint256

struct Proposal:
    name: String[100]
    voteCount: uint256

voters: public(HashMap[address, Voter])
proposals: public(HashMap[uint256, Proposal])
chairperson: public(address)
amountProposals: public(uint256)

MAX_NUM_PROPOSALS: constant(uint256) = 3

@external
def __init__():
    # The creator of the contract is the chairperson
    self.chairperson = msg.sender

@external
def addProposal(_proposalName: String[100]):
    # Access Control: Only the chairperson can add proposals
    assert msg.sender == self.chairperson
    i: uint256 = self.amountProposals
    self.proposals[i] = Proposal({
        name: _proposalName,
        voteCount: 0
    })
    self.amountProposals += 1

@external
def giveRightToVote(voter: address, _weight: uint256):
    assert msg.sender == self.chairperson
    assert not self.voters[voter].voted
    self.voters[voter].weight = _weight

@external
def vote(proposal: uint256):
    assert not self.voters[msg.sender].voted
    assert proposal < self.amountProposals

    self.voters[msg.sender].vote = proposal
    self.voters[msg.sender].voted = True

    self.proposals[proposal].voteCount += self.voters[msg.sender].weight
    self.voters[msg.sender].weight = 0

@view
@external
def winnerName() -> String[100]:
    winning_vote_count: uint256 = 0
    winning_proposal: uint256 = 0
    for i in range(MAX_NUM_PROPOSALS):
        if self.proposals[i].voteCount > winning_vote_count:
            winning_vote_count = self.proposals[i].voteCount
            winning_proposal = i
    return self.proposals[winning_proposal].name


Compile the contract to ensure our syntax is correct:

In [ ]:
!cd voting_app && ape compile

### 2. Testing the Voting Flow

Now we write a test. We want to:
1. Deploy the contract.
2. Add a few proposals.
3. Have the chairperson give voting rights to two different accounts.
4. Have those accounts cast votes.
5. Verify the winner.

In [ ]:
%%writefile voting_app/tests/test_voting_app.py
import pytest

@pytest.fixture
def chairperson(accounts):
    return accounts[0]

@pytest.fixture
def voter1(accounts):
    return accounts[1]

@pytest.fixture
def voter2(accounts):
    return accounts[2]

@pytest.fixture
def contract(project, chairperson):
    return project.DelegateVotingApp.deploy(sender=chairperson)

def test_voting_flow(contract, chairperson, voter1, voter2):
    # 1. Add proposals
    contract.addProposal("Proposal A", sender=chairperson)
    contract.addProposal("Proposal B", sender=chairperson)
    
    # 2. Give rights (voter 1 gets weight 1, voter 2 gets weight 2)
    contract.giveRightToVote(voter1, 1, sender=chairperson)
    contract.giveRightToVote(voter2, 2, sender=chairperson)
    
    # 3. Cast Votes
    # Voter 1 votes for Proposal A (index 0)
    contract.vote(0, sender=voter1)
    # Voter 2 votes for Proposal B (index 1)
    contract.vote(1, sender=voter2)
    
    # 4. Verify winner
    # Proposal B should win because Voter 2 had a weight of 2
    winner = contract.winnerName.call()
    assert winner == "Proposal B"


Let's run the tests and see if our democratic system works!

In [ ]:
!cd voting_app && ape test

**Congratulations!** You have completed the first part of this course. You've moved from understanding basic cryptography to deploying complex, tested smart contracts using professional Python tooling.